# RL basics: verifiable rewards, haiku edition

This tutorial uses Qwen3-4B and haiku poems to introduce the
**verifiable reward** pattern that underpins RL post-training:

1. Serve the base model.
2. Define a scoring function with a verifiable reward (syllable structure).
3. Evaluate the base model against that scorer.
4. GRPO-train the model with [slime](https://github.com/THUDM/slime) using the reward function.
5. Serve the trained checkpoint.
6. Evaluate it with the same scorer and compare.

**Why haikus?** A haiku has two attributes you can score
automatically — whether it follows the 5-7-5 syllable format
(deterministic, cheap) and whether the poem is actually good. That split between
*verifiable* and *subjective* rewards is exactly the landscape
RL post-training operates in. This tutorial covers the
verifiable half. In later tutorials, we will cover the subjective half.

In [1]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main nltk|

/bin/bash: -c: line 2: syntax error: unexpected end of file
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re

from modal_training_gym import (
    DeploymentConfig,
    EvalConfig,
    EvalRowResult,
    HuggingFaceDataset,
    Qwen3_4B,
    SlimeRecipe,
    TrainConfig,
    WandbConfig,
)
from modal_training_gym.common.checkpoint import list_checkpoints

## Serve the base model

So, how does Qwen3-4B currently fare at writing haikus? We can
serve the base model and find out.

`DeploymentConfig.serve()` builds and deploys a vLLM app, then
returns a `ModelDeployment` with the concrete endpoint URL.

In [3]:
base_model = Qwen3_4B()
base_model_deployment = DeploymentConfig(
    model=base_model,
).serve()
print(f"Base model deployed to {base_model_deployment.url}")

Base model deployed to https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct


Now that the model has come alive, we can request it to write a haiku about a topic.

In [4]:
response = base_model_deployment.generate(
    "Write a haiku about cat.",
    chat_template_kwargs={"enable_thinking": False},
)
print(response)

Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...


KeyboardInterrupt: 

Okay, how do we evaluate if that was a good haiku or not?
A haiku must follow the 5-7-5 syllable format.
We can count syllables using NLTK's CMU Pronouncing Dictionary
(with a regex fallback for words not in the dictionary)
and score how close each line is to its target.

In [13]:
_cmudict_cache = {}

def _get_cmudict() -> dict:
    if not _cmudict_cache:
        import nltk
        from nltk.corpus import cmudict
        nltk.download("cmudict", quiet=True)
        _cmudict_cache.update(cmudict.dict())
    return _cmudict_cache

def _count_syllables(text: str) -> int:
    cmu = _get_cmudict()
    total = 0
    for word in re.findall(r"[a-zA-Z]+", text):
        phones = cmu.get(word.lower())
        if phones:
            total += sum(p[-1].isdigit() for p in phones[0])
        else:
            count = len(re.findall(r"[aeiouy]+", word.lower()))
            if word.lower().endswith("e") and count > 1:
                count -= 1
            total += max(count, 1)
    return total

def score_haiku(response: str) -> float:
    lines = [line.strip() for line in response.strip().split("\n") if line.strip()]
    if len(lines) != 3:
        return -10
    total_diff = sum(
        abs(_count_syllables(line) - target)
        for line, target in zip(lines, [5, 7, 5])
    )
    return -float(total_diff)

In [6]:
response = base_model_deployment.generate(
    "Write a haiku about cat.",
    chat_template_kwargs={"enable_thinking": False},
)
print(response)
print(f"Score: {score_haiku(response)}")

Whiskers twitch in moonlight,  
tail sways soft through silent air—  
purr of midnight.
Score: -2.0


Let's also define a Haiku dataset.
Here, we use the statworx/haiku dataset from HuggingFace.
Each row has a `keywords` topic and a reference `text` haiku.
We can use this dataset to train our model.

In [6]:
class HaikuDataset(HuggingFaceDataset):
    hf_repo = "statworx/haiku"
    input_column = "keywords"
    output_column = "text"
    output_format = "jsonl"
    apply_chat_template = True
    system_prompt = (
        "You are a haiku poet. Write a haiku about the given topic. "
        "Use the 5-7-5 syllable format across three lines."
    )
    prompt_template = "Write a haiku about {input}."

train_dataset = HaikuDataset(n_rows=50)
eval_dataset = HaikuDataset(n_rows=10)

Let's take a look at the eval set. Each row has a `keywords`
topic and a reference `text` haiku.

In [7]:
df = eval_dataset.to_pandas()
print(len(df))
df.head(5)

/home/ec2-user/training-gym/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


10


,source,text,text_phonemes,keywords,keyword_phonemes,gruen_score,text_punc
0,bfbarry,Delicate savage. / You'll never hold the cinde...,deh|lax|kaxt sae|vaxjh / yuwl neh|ver hhowld d...,cinder,sihn|der,0.639071,NaN
1,bfbarry,A splash and a cry. / Words pulled from the ri...,ax splaesh aend ax kray / werdz puhld frahm dh...,the riverside,dhax rih|ver|sayd,0.563353,NaN
2,bfbarry,"Steamy, mist rising. / Rocks receiving downwar...",stiy|miy mihst ray|zaxng / raaks rax|siy|vaxng...,mist rising,mihst ray|zaxng,0.538326,NaN
3,bfbarry,You were broken glass. / But I touched you eve...,yuw wer brow|kaxn glaes / baht ay tahcht yuw i...,broken glass,brow|kaxn glaes,0.703446,NaN
4,bfbarry,Eyes dance with firelight. / The Moon and I ar...,ayz daens wihdh faxr|layt / dhax muwn aend ay ...,eyes dance,ayz daens,0.830985,NaN


Seems straightforward enough, right? How do we run an eval on our base model with this dataset?
We can transform our scoring function above into an Eval Configuration.

First, to explain, an Eval Configuration is a class that owns the model-calling loop.
The task-specific part is a scoring function passed to `.evaluate(...)`, which must
return `EvalRowResult`.

## Evaluate the base model

In [8]:
def eval_fn(_example: dict, response: str) -> EvalRowResult:
    return EvalRowResult(score=score_haiku(response), response=response)

eval_config = EvalConfig(
    dataset=eval_dataset,
    eval_fn=eval_fn,
    generate_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
)
print("——— Running base model evaluation... ———")
base_eval = eval_config.evaluate(base_model_deployment, debug=True)
print(f"Average haiku score: {base_eval.mean:.1f}")
print("——— Base model evaluation complete ———")

——— Running base model evaluation... ———
Waiting for https://modal-labs-joy-dev--qwen3-4b-serve-server.us-east.modal.direct (status 503)...


KeyboardInterrupt: 

## Train with slime

Now, let's actually train the model to write good haikus.
Here, we use the slime framework (https://github.com/THUDM/slime) on Modal.

In [14]:
async def haiku_rm(args, sample, **kwargs) -> float:
    return score_haiku(sample.response)

training_run = TrainConfig(
    model=base_model,
    dataset=train_dataset,
    recipe=SlimeRecipe(
        wandb=WandbConfig(project="gym-tutorial", group="qwen3-4b-haiku"),

        custom_rm_function=haiku_rm,

        num_rollout=10,
        save_interval=5,
        apply_chat_template_kwargs='{"enable_thinking": false}',

        image_overlay=lambda image: image.run_commands(
            "uv pip install --system aiohttp nltk>=3.8.0",
            "python -c \"import nltk; nltk.download('cmudict', quiet=True)\"",
        ),
    ),
)

## Train

`TrainConfig.train()` builds the Modal app, runs training, and
returns a `TrainResult` with the run ID and checkpoint path.

In [11]:
print("——— Running training... ———")
train_result = training_run.train()
print("——— Training complete ———")

——— Running training... ———
✓ Initialized. View run at 
https://modal.com/apps/modal-labs/joy-dev/ap-QHji1PVGchcMSIwcMLKBDg
⠋ Initializing...
⠸ Creating objects...objects...
└── ⠋ Creating mount PythonPackage:modal_training_gym: Finalizing index of 44 
⠦ Creating objects...
⠏ Creating objects...e:modal_training_gym
⠹ Creating objects...e:modal_training_gym
Building image im-oYBJlvHm0Hjg3ifOZnN0sym
⠸ Creating objects...
⠴ Creating objects...ackage:modal_training_gym
⠇ Creating objects...e:modal_training_gym
⠙ Creating objects...e:modal_training_gym
⠴ Creating objects...e:modal_training_gym
⠇ Creating objects...e:modal_training_gym
⠙ Creating objects...e:modal_training_gym
⠼ Creating objects...e:modal_training_gym
m🔨 Created mount PythonPackage:modal_training_gym
=> Step 0: FROM base
⠴ Creating objects...
m🔨 Created mount PythonPackage:modal_training_gym
=> Step 1: COPY . /
⠴ Creating objects...
⠧ Creating objects...ackage:modal_training_gym
⠋ Creating objects...e:modal_training_gym
⠸ Cr

[modal-client] 2026-05-07T08:24:55+0000 Timed out waiting for final app logs.


✓ App completed. View run at 
https://modal.com/apps/modal-labs/joy-dev/ap-QHji1PVGchcMSIwcMLKBDg
Training complete: slime-slimerecipe-1778140905
——— Training complete ———


## Serve and evaluate the trained checkpoint

The returned `TrainResult` has the checkpoint path and volume
metadata attached. You can pass an explicit `checkpoint=` to
`DeploymentConfig` to pin a specific checkpoint, or omit it to use
the model's default path.

In [10]:
checkpoint = list_checkpoints("slime-slimerecipe-1778140905")[-1]
print(checkpoint.path)

trained_model_deployment = DeploymentConfig(
    model=Qwen3_4B(),
    checkpoint=checkpoint,
    app_name="qwen3-4b-haiku-serve",
    served_model_name="qwen3-4b-haiku",
).serve()
print(f"Trained model deployed to {trained_model_deployment.url}")

/checkpoints/slime-slimerecipe-1778140905/iter_0000009_hf
Trained model deployed to https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct


## Evaluate the trained checkpoint

Now let's run the same eval on the trained model and compare.

In [13]:
print("——— Running trained model evaluation... ———")
trained_eval = eval_config.evaluate(trained_model_deployment, debug=True)
print(f"Trained haiku score: {trained_eval.mean:.1f}")
print("——— Trained model evaluation complete ———")

——— Running trained model evaluation... ———
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for https://modal-labs-joy-dev--qwen3-4b-haiku-serve-server.us-east.modal.direct (status 503)...
Waiting for

## Train off of a checkpoint
Hmm, looks like the trained model is not doing very well.
Maybe it's because it only trained for 10 iterations.

What happens if we train it for more?
We want to train it off of the latest checkpoint, not from scratch.

In [12]:
new_training_run = TrainConfig(
    model=base_model,
    dataset=train_dataset,    
    checkpoint=checkpoint,
    recipe=SlimeRecipe(
        wandb=WandbConfig(project="gym-tutorial", group="qwen3-4b-haiku"),

        custom_rm_function=haiku_rm,

        num_rollout=20,
        save_interval=10,
        apply_chat_template_kwargs='{"enable_thinking": false}',

        image_overlay=lambda image: image.run_commands(
            "uv pip install --system aiohttp nltk>=3.8.0",
            "python -c \"import nltk; nltk.download('cmudict', quiet=True)\"",
        ),
    ),
)
print("——— Running new training... ———")
new_train_result = new_training_run.train()
print("——— New training complete ———")

NameError: name 'haiku_rm' is not defined

## Evaluate the trained checkpoint

Now let's run the same eval on the newly trained model and compare.

In [9]:
new_checkpoint = list_checkpoints("slime-slimerecipe-1778140905")[-1]
print(new_checkpoint.path)

new_model_deployment = DeploymentConfig(
    model=Qwen3_4B(),
    checkpoint=new_checkpoint,
    app_name="qwen3-4b-haiku-serve-new",
    served_model_name="qwen3-4b-haiku",
).serve()
print(f"Newly trained model deployed to {new_model_deployment.url}")

/checkpoints/slime-slimerecipe-1778140905/iter_0000009_hf
Newly trained model deployed to https://modal-labs-joy-dev--qwen3-4b-haiku-serve-new-server.us-east.modal.direct


## Compare the results

Now let's compare the results of the newly trained model and the base model.

In [ ]:
print("——— Running trained model evaluation... ———")
new_eval = eval_config.evaluate(new_model_deployment, debug=True)
print(f"Trained model (new) haiku score: {new_eval.mean:.1f}")
print("——— Trained model (new) evaluation complete ———")

## Compare the results

Now let's compare the results of the newly trained model and the base model.

In [ ]:
print(f"Base model haiku score: {base_eval.mean:.1f}")
print(f"Trained model haiku score: {trained_eval.mean:.1f}")
print(f"Trained model (new) haiku score: {new_eval.mean:.1f}")